# Phase 5: Coding Domain Fine-Tuning (QLoRA)

**Dataset:** `HuggingFaceH4/CodeAlpaca_20K` — 20,111 code generation samples (full dataset)  
**Base model:** `mistralai/Mistral-7B-Instruct-v0.2`  
**Method:** QLoRA (4-bit quantization + LoRA adapters, r=16)  
**GPU required:** T4 (16GB) — available free on Kaggle  
**Estimated runtime:** ~90 minutes (3 epochs, ~19K training samples, seq_len=1024)

## What this notebook does
1. Loads and explores CodeAlpaca-20K
2. Filters samples that would be truncated mid-function (> 974 tokens)
3. Trains a QLoRA adapter with a coding system prompt
4. Evaluates with **HumanEval pass@1** — generates code, runs unit tests, reports accuracy
5. Pushes the adapter to HuggingFace Hub

## Why this eval metric matters
LLM-as-judge cannot reliably assess code correctness — it can't run the code.  
`pass@1` on HumanEval actually *executes* the generated solutions against unit tests.  
Baseline Mistral-7B-Instruct-v0.2: ~35–40% pass@1. Target after fine-tuning: ≥40%.

## CodeAlpaca field names
Unlike all prior phases which use `instruction`/`output`, CodeAlpaca uses `prompt`/`completion`.  
The formatter reads `sample["prompt"]` and `sample["completion"]` directly.

## 0. Kaggle Setup

Before running:
1. Enable GPU: **Settings → Accelerator → GPU T4 x1** (single GPU — do NOT use T4 x2)
2. Enable Internet: **Settings → Internet → On**
3. Add secret: **Add-ons → Secrets → HF_TOKEN**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## 1. Install Dependencies

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "bitsandbytes==0.45.3",
     "transformers==4.46.3",
     "peft==0.13.2",
     "trl==0.12.2",
     "accelerate==1.1.1",
     "datasets==3.0.0",
     "huggingface_hub",
     "human-eval",    # for pass@1 evaluation
     "tqdm",
    ],
    capture_output=True, text=True
)
print(result.stdout[-2000:] or "")
print(result.stderr[-500:] or "")
print("\nDone. Restart kernel if needed.")

In [ ]:
import transformers, peft, trl, bitsandbytes, accelerate, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")

## 2. Hyperparameters

In [ ]:
import torch

BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.2"
DATASET_NAME = "HuggingFaceH4/CodeAlpaca_20K"

# ── Dataset ─────────────────────────────────────────────────────────────────
MAX_SAMPLES     = None   # use full 20K — small enough to fit in one run
SHUFFLE_SEED    = 42
TEST_SIZE       = 0.05   # ~1K held-out; primary eval is HumanEval, not this split

# ── LoRA ────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Training ────────────────────────────────────────────────────────────────
OUTPUT_DIR          = "/kaggle/working/phase5-coding"
NUM_EPOCHS          = 3
BATCH_SIZE          = 4
GRAD_ACCUM_STEPS    = 4    # effective batch = 16
LEARNING_RATE       = 2.0e-4
MAX_SEQ_LENGTH      = 1024  # code completions need headroom vs finance (512)
LOGGING_STEPS       = 10
SAVE_STEPS          = 200

# ── HumanEval ───────────────────────────────────────────────────────────────
# 164 total problems. 30 = fast check (~5 min), 164 = full benchmark (~25 min)
HUMANEVAL_N_PROBLEMS = 30
HUMANEVAL_TIMEOUT    = 3.0   # seconds per unit test execution
PASS_AT_1_BASELINE   = "~35–40%"
PASS_AT_1_TARGET     = "≥40%"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" \
      if torch.cuda.is_available() else "")

## 3. Load and Explore the Dataset

CodeAlpaca uses `prompt` / `completion` — different from every prior phase.  
Content: Python-heavy but mixed (JS, Java, C++, Bash, SQL, etc.).

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET_NAME}...")
raw = load_dataset(DATASET_NAME, split="train")
print(f"Full dataset: {len(raw)} samples | Columns: {raw.column_names}")
print()

for i in range(3):
    s = raw[i]
    print(f"─── Sample {i} ───")
    print(f"PROMPT:     {s['prompt'][:200]}")
    print(f"COMPLETION: {s['completion'][:300]}")
    print()

In [ ]:
import numpy as np

# Token length distribution — critical for choosing MAX_SEQ_LENGTH
# Code completions are longer than MCQ or finance Q&A
lengths = [(len(s['prompt']) + len(s['completion'])) // 4 for s in raw]  # approx tokens
print(f"Approx token length (prompt + completion):")
print(f"  median: {int(np.median(lengths))}")
print(f"  p90:    {int(np.percentile(lengths, 90))}")
print(f"  p95:    {int(np.percentile(lengths, 95))}")
print(f"  p99:    {int(np.percentile(lengths, 99))}")
print(f"  max:    {max(lengths)}")
print(f"  MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} with 50-token buffer drops ~{sum(1 for l in lengths if l > MAX_SEQ_LENGTH-50)/len(lengths):.1%} of samples")

## 4. Load Tokenizer and Format Dataset

**Key design choice**: we pre-filter samples where `prompt + completion` would exceed `MAX_SEQ_LENGTH - 50` tokens.  
Truncating mid-function would teach the model to write incomplete code — worse than not seeing the sample at all.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
print(f"Tokenizer loaded. Vocab: {tokenizer.vocab_size}")

In [ ]:
CODING_SYSTEM = (
    "You are an expert software engineer and programmer. When given a coding task "
    "or programming problem, write clean, correct, and well-structured code. "
    "Follow best practices for the language being used. If the task is ambiguous, "
    "state your assumptions briefly before the code. Produce working solutions."
)


def filter_by_length(sample: dict) -> bool:
    """Drop samples where prompt+completion > MAX_SEQ_LENGTH - 50 tokens.
    Avoids truncating mid-function, which teaches incomplete code patterns."""
    n = len(tokenizer.encode(sample["prompt"] + sample["completion"]))
    return n <= MAX_SEQ_LENGTH - 50


def format_code_sample(sample: dict) -> str:
    """CodeAlpaca fields: {prompt, completion} — different from all prior phases.
    Preserving exact whitespace matters: indentation is semantically meaningful."""
    messages = [
        {"role": "system",    "content": CODING_SYSTEM},
        {"role": "user",      "content": sample["prompt"]},
        {"role": "assistant", "content": sample["completion"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


# Test
example = format_code_sample(raw[0])
print(f"Example (first 600 chars):\n{example[:600]}")
print(f"\nToken count: {len(tokenizer.encode(example))}")

In [ ]:
# 1. Length filter
before = len(raw)
raw_filtered = raw.filter(filter_by_length)
print(f"After length filter: {len(raw_filtered)} samples ({before - len(raw_filtered)} dropped)")

# 2. Shuffle
raw_filtered = raw_filtered.shuffle(seed=SHUFFLE_SEED)

# 3. Train/test split
split     = raw_filtered.train_test_split(test_size=TEST_SIZE, seed=42)
train_raw = split["train"]
test_raw  = split["test"]
print(f"Train: {len(train_raw)} | Test (held-out): {len(test_raw)}")

# 4. Format training set
train_dataset = train_raw.map(
    lambda s: {"text": format_code_sample(s)},
    remove_columns=train_raw.column_names,
)
print(f"Formatted training set: {len(train_dataset)} samples")

## 5. Load Model in 4-bit + Inject LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
import time
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    group_by_length=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(train_dataset) // eff_batch
print(f"Training {len(train_dataset)} samples × {NUM_EPOCHS} epochs")
print(f"Effective batch: {eff_batch} | Steps/epoch: ~{steps_per_epoch} | Total: ~{steps_per_epoch * NUM_EPOCHS}")

t0 = time.time()
trainer.train()
print(f"\nTraining time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
import os

adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter saved: {adapter_path} ({total_size / 1e6:.1f} MB)")

## 7. Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

log_history  = trainer.state.log_history
train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]

if train_losses:
    steps, losses = zip(*train_losses)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, color='steelblue', linewidth=1.5, alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Training Loss')
    ax.set_title('Phase 5: Coding QLoRA — Training Loss')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150)
    plt.show()
    print(f"Initial: {losses[0]:.4f} | Final: {losses[-1]:.4f} | Reduction: {(losses[0]-losses[-1])/losses[0]:.1%}")

## 8. Qualitative Code Generation Check

Before running HumanEval, manually inspect a few outputs to check:
- Syntactically valid Python
- Correct indentation
- Reasonable logic
- No hallucinated imports

In [ ]:
model.eval()

def generate_code(prompt: str, max_new_tokens: int = 512) -> str:
    """Greedy decoding for reproducible pass@1."""
    messages = [
        {"role": "system", "content": CODING_SYSTEM},
        {"role": "user",   "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs    = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,              # greedy — deterministic for pass@1
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


test_prompts = [
    "Write a Python function that returns the nth Fibonacci number using dynamic programming.",
    "Write a Python function that checks if a string is a palindrome (ignoring spaces and case).",
    "Write a Python function that flattens a nested list of arbitrary depth.",
]

for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    code = generate_code(prompt)
    print(f"Response:\n{code[:600]}")
    print()

## 9. HumanEval pass@1 Evaluation

HumanEval is a standard benchmark: 164 Python programming problems, each with a function signature + docstring and unit tests.

- **pass@1**: single greedy generation — does the code pass all unit tests?
- We use 30 problems (fast); set `HUMANEVAL_N_PROBLEMS = 164` for full benchmark
- Each problem gets a 3-second timeout for the unit test sandbox

**Note**: HumanEval is Python-only. CodeAlpaca trains on mixed languages — the pass@1 score captures Python quality but not JS/Java generalization.

In [ ]:
from human_eval.data import read_problems
from human_eval.execution import check_correctness
from tqdm.auto import tqdm
import json

problems = read_problems()
task_ids = list(problems.keys())[:HUMANEVAL_N_PROBLEMS]
print(f"Running HumanEval pass@1 on {HUMANEVAL_N_PROBLEMS} problems...")
print(f"(Set HUMANEVAL_N_PROBLEMS=164 for full benchmark)")
print()

passed  = 0
records = []

for task_id in tqdm(task_ids, desc="HumanEval"):
    problem    = problems[task_id]
    completion = generate_code(problem["prompt"])
    result     = check_correctness(problem, completion, timeout=HUMANEVAL_TIMEOUT)
    ok         = result["passed"]
    if ok:
        passed += 1
    records.append({
        "task_id":    task_id,
        "passed":     ok,
        "completion": completion[:300],
    })

pass_at_1 = passed / HUMANEVAL_N_PROBLEMS

print(f"\n{'='*50}")
print(f"HUMANEVAL PASS@1")
print(f"Problems: {HUMANEVAL_N_PROBLEMS} | Passed: {passed}")
print(f"pass@1 = {pass_at_1:.1%}")
print(f"Baseline (no fine-tuning): {PASS_AT_1_BASELINE}")
print(f"Target:                    {PASS_AT_1_TARGET}")
print(f"{'='*50}")

In [ ]:
# Per-problem breakdown — useful for debugging failure patterns
failed = [r for r in records if not r["passed"]]
print(f"\nFailed problems ({len(failed)}):")
for r in failed[:5]:
    print(f"  {r['task_id']}")
    print(f"    Completion start: {r['completion'][:150].replace(chr(10), ' ')}")

In [ ]:
eval_summary = {
    "phase":       "phase5_coding",
    "model":       BASE_MODEL,
    "adapter":     adapter_path,
    "dataset":     DATASET_NAME,
    "lora_r":      LORA_R,
    "num_epochs":  NUM_EPOCHS,
    "metric":      "pass@1",
    "n_problems":  HUMANEVAL_N_PROBLEMS,
    "n_passed":    passed,
    "pass_at_1":   round(pass_at_1, 4),
    "baseline":    PASS_AT_1_BASELINE,
    "target":      PASS_AT_1_TARGET,
}

with open(f"{OUTPUT_DIR}/eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)
print(json.dumps(eval_summary, indent=2))

## 10. Push Adapter to HuggingFace Hub

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or input("Enter HF_TOKEN: ").strip()

from huggingface_hub import login, HfApi
login(token=hf_token)

HF_USERNAME = "anksriv"
REPO_ID     = f"{HF_USERNAME}/mistral-7b-coding-qlora"

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)

trainer.model.push_to_hub(REPO_ID, token=hf_token)
tokenizer.push_to_hub(REPO_ID, token=hf_token)
api.upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/eval_results.json",
    path_in_repo="eval_results.json",
    repo_id=REPO_ID,
    token=hf_token,
)

print(f"\nPushed: https://huggingface.co/{REPO_ID}")

## 11. Summary

| Item | Value |
|------|-------|
| Base model | `mistralai/Mistral-7B-Instruct-v0.2` |
| Dataset | `HuggingFaceH4/CodeAlpaca_20K` (full 20K) |
| LoRA rank | r=16 |
| Epochs | 3 |
| Max seq length | 1024 |
| Eval metric | HumanEval pass@1 |
| Baseline | ~35–40% |
| Target | ≥40% |

**CodeAlpaca note**: The dataset contains some samples where the expected `completion` has bugs. These are CodeAlpaca dataset quality issues — not model failures. They reduce the training signal quality slightly but don't invalidate the overall adapter.

**HumanEval limitation**: Python-only benchmark. The CodeAlpaca training mix includes JS/Java/C++/Bash — cross-language generalization is real but not captured by pass@1.

**Next step:** All 5 adapters are trained. Open `app.py` to launch the multi-domain Gradio demo.